# 20.7 元学习 / Meta-Learning (Prototypical Networks & MAML)

**中文**:人看一两张"斑马"的照片就能认出所有斑马,而深度模型往往要成千上万张。**元学习(Meta-Learning),又叫"学会学习(learning to learn)"**,专门解决这个**少样本(few-shot)** 难题:训练时不是学"某个任务",而是学**跨很多任务的"快速学习能力"**——于是面对一个**全新任务**,只需极少样本就能学会。核心范式是**回合式(episodic)训练**:每个回合模拟一个"N 类 K 样本(N-way K-shot)"的小任务。本节从零实现两大流派:**原型网络(度量学习)** 和 **MAML(优化学习)**。
**English**: A human recognizes all zebras from one or two photos, while deep models often need thousands. **Meta-learning, aka "learning to learn,"** tackles this **few-shot** problem: instead of learning "a task," it learns a **"fast-learning ability" across many tasks** — so faced with a **brand-new task**, it learns from very few examples. The core paradigm is **episodic training**: each episode simulates a small "N-way K-shot" task. This section implements two schools from scratch: **Prototypical Networks (metric-based)** and **MAML (optimization-based)**.

---

**中文**:关键概念:
**English**: Key concepts:
- **N-way K-shot**:一个少样本任务 = 从 N 个类里、每类给 K 个带标签的**支持样本(support set)**,要给**查询样本(query set)** 分类。5-way 5-shot = 5 个新类、每类 5 张、认新图。
  **N-way K-shot**: a few-shot task = N classes, K labeled **support** examples each, classify the **query set**. 5-way 5-shot = 5 new classes, 5 images each, classify new images.
- **元训练 vs 元测试**:在**一批类**上做很多回合的元训练,学到"快速适应"的能力;在**完全不同的一批类**上元测试(验证泛化到新任务)。

  **Meta-train vs meta-test**: many episodes of meta-training on **one set of classes** to learn "fast adaptation," then meta-test on a **completely different set of classes** (verifying generalization to new tasks).

**中文**:两大流派:
**English**: Two schools:
- **① 原型网络(Prototypical Networks,度量学习)**:学一个**嵌入函数**,把样本映射到一个空间;每个类用其支持样本嵌入的**均值当"原型"**;查询样本**归到最近的原型**。思想极简:*"学一个好的度量空间,同类聚在一起,分类就是找最近的类中心。"* 简单、稳、强。
  **Prototypical Networks (metric-based)**: learn an **embedding** mapping samples to a space; each class's **prototype = the mean of its support embeddings**; classify a query by the **nearest prototype**. Minimalist idea: *"learn a good metric space where same-class points cluster, and classification is just finding the nearest class center."* Simple, robust, strong.
- **② MAML(Model-Agnostic Meta-Learning,优化学习)**:学一个**好的初始化参数**,使得对任意新任务,**只需几步梯度下降**就能收敛到好解。它是**双层优化**:内循环在单个任务的支持集上做几步梯度、外循环优化"初始化"使得内循环之后在查询集上表现好。*"学一个'离所有任务都很近'的起点。"*
  **MAML (Model-Agnostic Meta-Learning, optimization-based)**: learn a **good initialization** such that for any new task, **a few gradient steps** suffice to reach a good solution. It is **bi-level optimization**: the inner loop takes a few gradient steps on a task's support set, and the outer loop optimizes the "initialization" so that after the inner loop it performs well on the query set. *"Learn a starting point close to all tasks."*

> 💡 **面试速查 / Interview cheat-sheet（★★ 少样本必考）**
> **中文**:元学习=**学会学习**, 解决少样本(N-way K-shot)。**回合式训练**:元训练学"快速适应", 元测试用全新类验证。两大流派:①**原型网络(度量)**:学嵌入, 类原型=支持集均值, 查询归最近原型(简单稳);②**MAML(优化)**:学一个好初始化, 新任务几步梯度就适应, 双层优化(内循环适应+外循环优化初始化, 需二阶梯度, 有一阶近似 FOMAML/Reptile)。其他:Matching Net、Relation Net。**vs 迁移学习/微调**:迁移是"预训练大模型+微调", 元学习是"专门学'快速适应'的能力", 极少样本时更优;但现在**大模型的 in-context learning(上下文学习)** 某种意义上是元学习的胜利(prompt 里给几个例子就学会=few-shot)。用途:少样本图像/文本分类、机器人快速适应新环境、冷启动、个性化。
> **English**: Meta-learning = **learning to learn**, for few-shot (N-way K-shot). **Episodic training**: meta-train to learn "fast adaptation," meta-test on brand-new classes. Two schools: ① **Prototypical Networks (metric)**: learn an embedding, class prototype = support mean, classify by nearest prototype (simple, robust); ② **MAML (optimization)**: learn a good initialization so a new task adapts in a few gradient steps — bi-level optimization (inner loop adapts + outer loop optimizes the init, needs second-order gradients; first-order approximations FOMAML/Reptile exist). Others: Matching Net, Relation Net. **vs transfer learning/fine-tuning**: transfer is "pretrain a big model + fine-tune," meta-learning specifically learns "fast-adaptation ability," better with extremely few samples; but modern **LLM in-context learning** is in a sense meta-learning's triumph (give a few examples in the prompt and it learns = few-shot). Uses: few-shot image/text classification, robots adapting to new environments, cold-start, personalization.


In [ ]:

# ============================================================
# ① 原型网络:少样本分类 / Prototypical Networks: few-shot classification
# 中文:合成任务——每个类由 4 个"信号维"定义, 但观测里混入 30 个大方差"干扰维"(与类无关)。
#      原始最近邻会被干扰维淹没; 元学习出的嵌入能学会'忽略干扰、聚焦信号', 泛化到全新类。
# English: synthetic — each class defined by 4 "signal dims", but observations mix in 30 large-variance
#      "nuisance dims" (class-irrelevant). Raw nearest-centroid drowns in nuisance; a meta-learned embedding
#      learns to ignore nuisance and generalizes to brand-new classes.
# ============================================================
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, time, matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0); rng=np.random.default_rng(0)
D_SIG, D_NUIS = 4, 30; D_OBS=D_SIG+D_NUIS
def make_center(): return rng.normal(0,2,D_SIG)                          # 类中心(只在信号维)/ class center
def sample_class(center, k):
    sig=center + rng.normal(0,0.3,(k,D_SIG))                             # 信号:类相关 / signal (class-relevant)
    nuis=rng.normal(0,3,(k,D_NUIS))                                      # 干扰:大方差, 类无关 / nuisance
    return np.hstack([sig,nuis]).astype(np.float32)
def make_episode(nway=5, kshot=5, nquery=5):
    centers=[make_center() for _ in range(nway)]; sup=[];qry=[];ys=[];yq=[]
    for c in range(nway):
        sup.append(sample_class(centers[c],kshot)); qry.append(sample_class(centers[c],nquery))
        ys+=[c]*kshot; yq+=[c]*nquery
    return (torch.tensor(np.vstack(sup)),torch.tensor(ys),torch.tensor(np.vstack(qry)),torch.tensor(yq),nway)

class Embedding(nn.Module):
    def __init__(s): super().__init__(); s.net=nn.Sequential(nn.Linear(D_OBS,64),nn.ReLU(),nn.Linear(64,16))
    def forward(s,x): return s.net(x)

def proto_loss_acc(emb, ep):
    sup,ys,qry,yq,nway=ep; zs=emb(sup); zq=emb(qry)
    prototypes=torch.stack([zs[ys==c].mean(0) for c in range(nway)])    # 类原型=支持嵌入均值 / prototypes
    logits=-((zq[:,None,:]-prototypes[None,:,:])**2).sum(2)             # 到各原型的负距离 / -distance
    return F.cross_entropy(logits,yq), (logits.argmax(1)==yq).float().mean().item()

emb=Embedding(); opt=torch.optim.Adam(emb.parameters(),1e-3)
t=time.time()
for it in range(2000):                                                  # 每回合一个新的少样本任务 / episodic
    loss,_=proto_loss_acc(emb, make_episode()); opt.zero_grad(); loss.backward(); opt.step()
proto_acc=np.mean([proto_loss_acc(emb, make_episode())[1] for _ in range(300)])   # 元测试:全新类 / meta-test
# 基线:原始特征最近质心(不学嵌入)/ baseline: raw nearest-centroid (no learning)
def raw_nn_acc(ep):
    sup,ys,qry,yq,nway=ep; protos=torch.stack([sup[ys==c].mean(0) for c in range(nway)])
    return (((qry[:,None,:]-protos[None,:,:])**2).sum(2).argmin(1)==yq).float().mean().item()
raw_acc=np.mean([raw_nn_acc(make_episode()) for _ in range(300)])
print(f"原型网络(学到的嵌入)5-way 5-shot 准确率 / ProtoNet: {proto_acc:.3f}  ({time.time()-t:.0f}s)")
print(f"原始最近质心(无元学习)/ raw nearest-centroid: {raw_acc:.3f}")
print("→ 元学习出的嵌入学会忽略干扰维、聚焦信号维, 泛化到从没见过的类 / learned to ignore nuisance")


**中文**:原型网络在**从没见过的新类**上达到 ~0.99,而原始最近质心只有 ~0.38——因为 30 个大方差干扰维淹没了原始欧氏距离,但元学习出的嵌入学会了"**忽略干扰、只看信号**",而且这个能力泛化到了全新的类。这就是"学会学习"。

现在实现 **② MAML**,用它经典的**正弦回归**任务:每个任务是一条随机振幅/相位的正弦曲线,给 5 个点,要拟合整条曲线。MAML 学一个初始化,使得对任意新正弦**只需 5 步梯度**就能拟合好。
**English**: Prototypical Networks reach ~0.99 on **never-seen new classes**, while raw nearest-centroid is only ~0.38 — the 30 large-variance nuisance dims drown the raw Euclidean distance, but the meta-learned embedding learned to "**ignore nuisance, focus on signal**," and this ability generalizes to brand-new classes. That is "learning to learn."

Now implement **② MAML** on its classic **sinusoid regression** task: each task is a sine of random amplitude/phase; given 5 points, fit the whole curve. MAML learns an initialization such that any new sine can be fit in **just 5 gradient steps**.


In [ ]:

# ============================================================
# ② MAML:正弦回归 / MAML on sinusoid regression
# ============================================================
def sample_sine(seed=None):
    r=np.random.default_rng(seed); return r.uniform(0.1,5.0), r.uniform(0,np.pi)   # 振幅, 相位 / amplitude, phase
def sine_data(A,ph,k,seed=None):
    r=np.random.default_rng(seed); x=r.uniform(-5,5,(k,1)).astype(np.float32)
    return torch.tensor(x), torch.tensor((A*np.sin(x+ph)).astype(np.float32))
class SineNet(nn.Module):
    def __init__(s): super().__init__(); s.net=nn.Sequential(nn.Linear(1,40),nn.ReLU(),nn.Linear(40,40),nn.ReLU(),nn.Linear(40,1))
    def forward(s,x): return s.net(x)
def func_forward(x, weights):                                # 用给定权重前向(便于内循环手动更新)/ functional forward
    h=x
    for i in range(0,len(weights)-2,2): h=torch.relu(F.linear(h,weights[i],weights[i+1]))
    return F.linear(h,weights[-2],weights[-1])

net=SineNet(); meta_opt=torch.optim.Adam(net.parameters(),1e-3); inner_lr=0.01; K=5
t=time.time()
for it in range(3000):
    meta_loss=0.0
    for _ in range(4):                                       # 一个 meta-batch 的任务 / meta-batch of tasks
        A,ph=sample_sine(); xs,ys=sine_data(A,ph,K); xq,yq=sine_data(A,ph,K)
        w=list(net.parameters())
        pred=func_forward(xs,w); loss=((pred-ys)**2).mean()  # 内循环:在支持集上算梯度 / inner: support loss
        grads=torch.autograd.grad(loss, w, create_graph=True)# 保留计算图做二阶 / second-order
        w_adapted=[p-inner_lr*g for p,g in zip(w,grads)]     # 内循环一步适应 / one adaptation step
        predq=func_forward(xq,w_adapted); meta_loss=meta_loss+((predq-yq)**2).mean()  # 外循环:适应后的查询损失
    meta_loss=meta_loss/4
    meta_opt.zero_grad(); meta_loss.backward(); meta_opt.step()   # 外循环:优化初始化 / optimize init
print(f"MAML 元训练完成 / meta-trained ({time.time()-t:.0f}s)")

# 元测试:一个全新正弦, 只给5个点, 适应几步 / meta-test: adapt to a NEW sine from 5 points
A,ph=sample_sine(seed=99); xs,ys=sine_data(A,ph,K,seed=100)
xtest=torch.tensor(np.linspace(-5,5,100).reshape(-1,1).astype(np.float32)); ytrue=A*np.sin(xtest.numpy()+ph)
w=[p.detach().clone().requires_grad_(True) for p in net.parameters()]
mse_before=((func_forward(xtest,w).detach().numpy()-ytrue)**2).mean()
preds_over_steps=[func_forward(xtest,w).detach().numpy()]
for step in range(5):                                        # 5 步梯度适应 / 5 adaptation steps
    loss=((func_forward(xs,w)-ys)**2).mean(); grads=torch.autograd.grad(loss,w)
    w=[(p-inner_lr*g).detach().requires_grad_(True) for p,g in zip(w,grads)]
    preds_over_steps.append(func_forward(xtest,w).detach().numpy())
mse_after=((preds_over_steps[-1]-ytrue)**2).mean()
print(f"新正弦(5个点): 适应前 MSE {mse_before:.2f} → 5步梯度后 {mse_after:.2f} (振幅 A={A:.1f})")
print("→ MAML 初始化让模型只用5个点、5步梯度就拟合出全新正弦 / adapts to a new sine in 5 steps from 5 points")


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(17,4.7))
# ① ProtoNet vs 原始最近质心 / ProtoNet vs raw
ax[0].bar(["原始最近质心\nraw (no ML)","原型网络\nProtoNet"],[raw_acc,proto_acc],color=["#C44E52","#4C72B0"])
ax[0].axhline(0.2,ls="--",color="gray",label="随机 5-way=0.2"); ax[0].set_ylim(0,1.05)
for i,v in enumerate([raw_acc,proto_acc]): ax[0].text(i,v,f"{v:.2f}",ha="center",va="bottom")
ax[0].set_title("原型网络泛化到新类 / ProtoNet on new classes"); ax[0].set_ylabel("5-way 5-shot 准确率"); ax[0].legend(fontsize=8)
# ② MAML 适应过程 / MAML adaptation over steps
ax[1].plot(xtest.numpy(),ytrue,"g-",lw=2,label="真实新正弦 true")
ax[1].scatter(xs.numpy(),ys.numpy(),c="k",s=60,zorder=5,label="仅5个点 5 shots")
for step,al in [(0,0.35),(1,0.6),(5,0.95)]:
    ax[1].plot(xtest.numpy(),preds_over_steps[step],"--",alpha=al,label=f"{step}步适应")
ax[1].set_title("MAML:5个点+几步梯度拟合新正弦 / adapt from 5 points"); ax[1].set_xlabel("x"); ax[1].legend(fontsize=7)
# ③ MAML 每步的 MSE 下降 / MAML MSE over adaptation steps
mses=[((p-ytrue)**2).mean() for p in preds_over_steps]
ax[2].plot(range(len(mses)),mses,"o-",color="#4C72B0")
ax[2].set_title("适应步数 vs 误差:几步就收敛 / MSE drops in a few steps"); ax[2].set_xlabel("梯度适应步数"); ax[2].set_ylabel("MSE(整条曲线)")
plt.tight_layout(); plt.savefig("/tmp/adv07_viz.png",dpi=80); plt.show()
print("原型网络学好度量空间、MAML 学好初始化——两条路都实现了'少样本快速学习'")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **两条路殊途同归:都在"学会快速学习"**:原型网络学一个**好的度量空间**(同类聚一起),于是新任务只需算几个类中心、找最近的;MAML 学一个**好的初始化**,于是新任务只需几步梯度。关键都在于——它们**在很多任务上元训练**,学到的不是某个任务的解,而是"**面对新任务如何快速学会**"的能力。ProtoNet 在全新类上 0.99(原始 0.38)、MAML 用 5 个点几步梯度拟合全新正弦,都印证了这一点。
2. **元学习为什么有效**:回合式训练**逼模型面对源源不断的新任务**,不能靠死记硬背某个任务,只能学"通用的适应策略"。原型网络因此学会"忽略无关的干扰维、只看类间真正区分的信号维"(这个能力对新类照样有效);MAML 学到的初始化"离所有正弦任务都近",所以往任何方向走几步都能到。
3. **诚实的现实与演进**:①**元学习实现有坑**——MAML 要算**二阶梯度**(对梯度再求梯度),内存大、易不稳,实践中常用一阶近似(FOMAML)或 Reptile;②**度量法(ProtoNet)更简单稳健**,是少样本分类的强基线, 往往比 MAML 更实用;③**和迁移学习/微调比,元学习不总是赢**——在有大规模预训练模型时,"预训练+微调"经常打平甚至超过专门的元学习(这也是学界的诚实共识);④**最深刻的转折**:今天大模型的 **in-context learning(上下文学习)**——你在 prompt 里给几个例子,模型当场学会——本质就是元学习的终极形态,只不过"元训练"发生在海量文本的预训练里。从这个角度,**元学习的思想赢了,只是赢的方式和当年预想的不同。**

**English**:
1. **Two paths, one goal: "learning to learn fast"**: Prototypical Networks learn a **good metric space** (same-class points cluster), so a new task just computes a few class centers and finds the nearest; MAML learns a **good initialization**, so a new task just takes a few gradient steps. The key for both — they **meta-train over many tasks**, learning not a task's solution but the ability to "**quickly learn a new task**." ProtoNet's 0.99 on new classes (raw 0.38) and MAML's fitting a brand-new sine from 5 points confirm this.
2. **Why meta-learning works**: episodic training **forces the model to face an endless stream of new tasks**, so it can't memorize a single task and must learn a "general adaptation strategy." Prototypical Networks thus learn to "ignore irrelevant nuisance dims and focus on the truly discriminative signal dims" (an ability that transfers to new classes too); MAML's init sits "close to all sine tasks," so a few steps in any direction reach a good solution.
3. **Honest reality and evolution**: ① **meta-learning has implementation pitfalls** — MAML needs **second-order gradients** (gradients of gradients), memory-heavy and unstable, so practice often uses first-order approximations (FOMAML) or Reptile; ② **metric methods (ProtoNet) are simpler and more robust**, a strong baseline for few-shot classification, often more practical than MAML; ③ **vs transfer learning/fine-tuning, meta-learning doesn't always win** — with large pretrained models, "pretrain + fine-tune" often matches or beats dedicated meta-learning (an honest academic consensus); ④ **the deepest twist**: today's LLM **in-context learning** — you give a few examples in the prompt and the model learns on the spot — is essentially meta-learning's ultimate form, except the "meta-training" happened in massive-text pretraining. In that sense, **meta-learning's idea won, just not in the way originally envisioned.**

> 💼 **实战视角 / Practical angle**
> **中文**:元学习/少样本的实战:①**少样本图像/文本分类**(新品类只有几张图);②**冷启动**(新用户/新物品少数据);③**机器人**快速适应新环境/新任务;④**药物/材料**(实验数据极少)。落地经验:①先试**预训练+微调**(有大模型时通常最强、最省事);②少样本分类首选 **原型网络/度量学习**(比 MAML 稳);③MAML 用**一阶近似(FOMAML/Reptile)** 省算力;④现在很多"少样本"直接用 **LLM/多模态大模型的 in-context learning 或 few-shot prompting**。面试金句:*"元学习='学会学习', 回合式训练跨任务学'快速适应'; 原型网络学度量空间(新类找最近原型), MAML 学好初始化(几步梯度适应新任务, 双层优化需二阶梯度); 少样本首选度量法, 但大模型的 in-context learning 是元学习思想的现代胜利。"*
> **English**: Meta-learning / few-shot in practice: ① **few-shot image/text classification** (a new category with only a few images); ② **cold-start** (new users/items with little data); ③ **robots** adapting quickly to new environments/tasks; ④ **drug/material discovery** (very little experimental data). Practical lessons: ① try **pretrain + fine-tune** first (usually strongest and simplest with big models); ② for few-shot classification prefer **Prototypical Networks/metric learning** (more robust than MAML); ③ use **first-order approximations (FOMAML/Reptile)** to save compute for MAML; ④ many "few-shot" tasks now just use **LLM/multimodal in-context learning or few-shot prompting**. Interview line: *"Meta-learning = learning to learn; episodic training learns 'fast adaptation' across tasks; Prototypical Networks learn a metric space (nearest prototype for new classes), MAML learns a good initialization (a few gradient steps adapt to a new task, bi-level optimization needing second-order gradients); prefer metric methods for few-shot, but LLM in-context learning is meta-learning's modern triumph."*

---
### 小结 / Summary
- **中文**:元学习="学会学习", 回合式训练跨任务学"快速适应"; N-way K-shot; 元测试用全新类。
- **English**: Meta-learning = "learning to learn," episodic training learns "fast adaptation" across tasks; N-way K-shot; meta-test on brand-new classes.
- **中文**:原型网络(度量):学嵌入+最近原型(新类0.99 vs 原始0.38); MAML(优化):学好初始化, 几步梯度适应新任务。
- **English**: Prototypical Networks (metric): learn an embedding + nearest prototype (0.99 on new classes vs 0.38 raw); MAML (optimization): learn a good init, adapt in a few gradient steps.
- **中文**:度量法更稳; MAML 需二阶梯度(用 FOMAML/Reptile); 大模型 in-context learning 是元学习思想的现代胜利。
- **English**: Metric methods are more robust; MAML needs second-order gradients (use FOMAML/Reptile); LLM in-context learning is meta-learning's modern triumph.
